# Data

In [126]:
import nltk
from nltk.corpus import gutenberg, stopwords
from nltk import tokenize
import autocorrect
from nltk.stem import WordNetLemmatizer
from autocorrect import Speller
nltk.download('omw-1.4')
import regex
from gensim.models import Word2Vec
import gensim.downloader as api
import pandas as pd


[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Instructions
Objective:
To train word embeddings on famous works of Shakespeare and evaluate their understanding.

Data:
The entire text of plays: 1) The Tragedy of Hamlet, Prince of Denmark, 2) The Tragedy of Macbeth, and 3) The Tragedy of Julius Caesar. These are available from the Gutenberg corpus of the NLTK library. Characters and synopses can be found on Wikipedia.

Problem Statement:
Natural language processing is an important part of the most advanced artificial intelligence software we have today. By studying volumes of text, word embeddings are able to elicit meaning from the words within training data. Your goal is to train a word embedding on three famous works of Shakespeare to determine how well your embedding can understand the meaning of character names and other Shakespearean English words found in these plays.

Steps to be completed:
Create a Jupyter notebook and complete the following steps:



# Data
Use nltk.corpus.gutenberg.raw to load the three plays listed above into a single variable and lower the case.


In [127]:
# Assigning the plays to variables
nltk.download('gutenberg')
hamlet = gutenberg.raw('shakespeare-hamlet.txt')
macbeth = gutenberg.raw('shakespeare-macbeth.txt')
julius_caesar = gutenberg.raw('shakespeare-caesar.txt')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!



Tokenize the text into sentences, and then each sentence into words.


In [128]:
# Keeping all the text in a single variable with normalized case
lower_text = (hamlet + macbeth + julius_caesar).lower()

In [129]:
# Tokenizing sentences
sentences = tokenize.sent_tokenize(lower_text)

In [130]:
# Tokenizing words
words = [tokenize.word_tokenize(sent) for sent in sentences]

Use Speller from the autocorrect library to correct spelling mistakes. 


In [131]:
# Correcting spelling errors
spell = Speller(lang = 'en')
correct_sentences = [[spell(word) for word in sentence] for sentence in words]

Create a list of stopwords (using publicly available lists and/or adding your own) and remove these.


In [132]:
# Removing stop words
stop_words = set(stopwords.words('english'))
no_stop_words = [[word for word in sentence if word not in stop_words] for sentence in correct_sentences]

Use regular expressions (the re library) to do any additional cleanup of the text you wish to do.


Use PorterStemmer or WordNetLemmatizer from nltk.stem on the text.


In [133]:
# Lemmatizing
lemmatizer = WordNetLemmatizer()
lemmatized_words = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in no_stop_words]

In [134]:
# Removing punctuation and then the empty tokens created by the cleanup
regex_cleaned = [[regex.sub(r'[^a-zA-Z0-9]', '', word) for word in sentence] for sentence in lemmatized_words]
regex_cleaned = [[word for word in sentence if word.strip() != ''] for sentence in regex_cleaned]

Print out the words in the first five sentences of the processed text data. (Viewing this may give you additional ideas for the previous steps.)


In [135]:
# Printing the first five sentences
for i in range(5):
    print(f"Sentence {i}: {regex_cleaned[i]}")

Sentence 0: ['tragedy', 'hamlet', 'william', 'shakespeare', '1599', 'act', 'prime']
Sentence 1: ['scene', 'prima']
Sentence 2: ['enter', 'bernard', 'francisco', 'two', 'sentinel']
Sentence 3: ['bernard']
Sentence 4: ['s']


In [136]:
# Comparing with the actual few sentences
for i in range(5):
    print(f"Sentence {i}: {words[i]}")

Sentence 0: ['[', 'the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', '1599', ']', 'actus', 'primus', '.']
Sentence 1: ['scoena', 'prima', '.']
Sentence 2: ['enter', 'barnardo', 'and', 'francisco', 'two', 'centinels', '.']
Sentence 3: ['barnardo', '.']
Sentence 4: ['who', "'s", 'there', '?']



# Modeling 
Create a CBOW word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data. Use gensim.model.wv.key_to_index  and gensim.model.wv.get_vecattr to print out a list of the 20 most frequent words in the vocabulary along with the word count. Consider improving the text cleaning steps above based on this information. 


# Processing

## Word2Vec models

### CBOW: target word from context words

In [137]:
model = Word2Vec(sentences=regex_cleaned, vector_size=100, window=5, min_count=3, workers=4, sg=0, epochs=20)

In [138]:
top20 = list(model.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20:
    count = model.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

d                585
ham              337
thou             307
lord             306
shall            300
s                295
come             284
king             248
enter            230
good             221
let              220
mac              205
thy              202
like             200
cesar            193
one              188
make             185
know             184
v                175
thee             174


Based on this word count, we notice a few things:
- d is the top word, which possibly came from words I'd, you'd, we'd. It is just a contraction of 'would', and should probably be removed. Similarly, s and v perhaps correspond to 'is' and 'have'.
- Archaic pronouns like 'thee', 'thy', 'thou' constitute stop words and should also be removed.
- 'Caesar', 'Hamlet' and 'Macbeth' being correct to 'cesar', 'mac', 'ham' sounds lowkey appetizing and defeats the purpose. However, we cannot remove the spelling correction step because the text consists of Latin words which should be converted to English. A better decision might be to manually code those words to preserve the original meaning.

In [139]:
archaic = {"thou", "thee", "thy", "thine", "ye", "hath", "doth"}
stop_words.update(archaic)

In [140]:
regex_cleaned = [[w for w in sent if len(w) > 1] for sent in regex_cleaned]

In [141]:
name_map = {
    "hamlet": "hamlet",
    "ham": "hamlet",
    "macbeth": "macbeth",
    "mac": "macbeth",
    "caesar": "caesar",
    "cesar": "caesar"
}

normalized = [[name_map.get(w, w) for w in sent] for sent in regex_cleaned]

In [142]:
for i in range(5):
    print(f"Sentence {i}: {normalized[i]}")

Sentence 0: ['tragedy', 'hamlet', 'william', 'shakespeare', '1599', 'act', 'prime']
Sentence 1: ['scene', 'prima']
Sentence 2: ['enter', 'bernard', 'francisco', 'two', 'sentinel']
Sentence 3: ['bernard']
Sentence 4: []


In [143]:
model = Word2Vec(sentences=normalized, vector_size=100, window=5, min_count=3, workers=4, sg=0, epochs=20)

In [144]:
top20 = list(model.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20:
    count = model.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

hamlet           444
thou             307
lord             306
shall            300
come             284
macbeth          271
king             248
enter            230
good             221
let              220
thy              202
like             200
caesar           193
one              188
make             185
know             184
thee             174
self             166
would            163
aboutus          162


In [145]:
top_5 = ['hamlet', 'cauldron', 'nature', 'spirit', 'general', 'prythee']
for i in top_5:
    print(f'Most similar results for: {i}')
    print(model.wv.most_similar(i, topn=5))


Most similar results for: hamlet
[('good', 0.9929826259613037), ('polo', 0.9924790263175964), ('sweet', 0.98771071434021), ('attendant', 0.9875280261039734), ('ratio', 0.987463653087616)]
Most similar results for: cauldron
[('opinion', 0.9987410306930542), ('share', 0.9986475110054016), ('burned', 0.9986176490783691), ('tear', 0.9986040592193604), ('enough', 0.9985946416854858)]
Most similar results for: nature
[('whose', 0.9976707696914673), ('though', 0.9974689483642578), ('even', 0.9974145293235779), ('judgement', 0.9974014759063721), ('act', 0.9973229169845581)]
Most similar results for: spirit
[('hide', 0.9980875849723816), ('wound', 0.9978452324867249), ('vse', 0.9977707266807556), ('bone', 0.9977434873580933), ('even', 0.9976552128791809)]
Most similar results for: general
[('vile', 0.9988468289375305), ('enemy', 0.9987438321113586), ('cyber', 0.998701810836792), ('hill', 0.9986593723297119), ('speaker', 0.998654305934906)]
Most similar results for: prythee
[('farewell', 0.99808

Create a skipgram word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data.


In [146]:
model_sg = Word2Vec(sentences=normalized, vector_size=100, window=5, min_count=3, workers=4, sg=1, epochs=20)

In [147]:
top20_sg = list(model_sg.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20_sg:
    count = model_sg.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

hamlet           444
thou             307
lord             306
shall            300
come             284
macbeth          271
king             248
enter            230
good             221
let              220
thy              202
like             200
caesar           193
one              188
make             185
know             184
thee             174
self             166
would            163
aboutus          162


In [148]:
top_5 = ['hamlet', 'thou', 'lord', 'shall', 'come']
for i in top_5:
    print(f'Most similar results for: {i}')
    print(model_sg.wv.most_similar(i, topn=5))

Most similar results for: hamlet
[('polo', 0.9037594199180603), ('lord', 0.8538440465927124), ('faith', 0.8498649597167969), ('ophelia', 0.8321051597595215), ('alert', 0.8309140205383301)]
Most similar results for: thou
[('art', 0.7679350972175598), ('shalt', 0.7639899253845215), ('didst', 0.7618868350982666), ('wouldst', 0.7571813464164734), ('fellow', 0.7466831207275391)]
Most similar results for: lord
[('hamlet', 0.8538439273834229), ('polo', 0.8277920484542847), ('ophelia', 0.7654093503952026), ('faith', 0.7630994319915771), ('queen', 0.7581347823143005)]
Most similar results for: shall
[('satisfied', 0.824519693851471), ('receive', 0.8162141442298889), ('elder', 0.8138745427131653), ('permission', 0.810947060585022), ('senate', 0.8106380701065063)]
Most similar results for: come
[('either', 0.8287853598594666), ('sit', 0.8286930918693542), ('dunsinane', 0.8265745043754578), ('fetch', 0.8161941170692444), ('wood', 0.799146294593811)]


Load the pretrained GloVe model from gensim.models.keyedvectors for comparison with the models trained on Shakespeare text. Use markdown to make note of the data that GloVe has been trained on.

GloVe Training Data
GloVe models available via gensim.downloader were trained on massive general-domain corpora like Wikipedia 2014 + Gigaword 5 (6B tokens, 400K vocab for the 100d version).

In [149]:
# Recommended: glove-wiki-gigaword-100 (matches your vector_size=100)
glove_model = api.load("glove-wiki-gigaword-100")


In [150]:

# Now compare, e.g., on "oeuvre"
print("Skip-gram (Shakespeare):", model_sg.wv.most_similar("hamlet", topn=5))
print("GloVe:", glove_model.most_similar("hamlet", topn=5))

Skip-gram (Shakespeare): [('polo', 0.9037594199180603), ('lord', 0.8538440465927124), ('faith', 0.8498649597167969), ('ophelia', 0.8321051597595215), ('alert', 0.8309140205383301)]
GloVe: [('village', 0.6998987197875977), ('town', 0.6558532118797302), ('situated', 0.5926076769828796), ('located', 0.5660547614097595), ('unincorporated', 0.5599358677864075)]


# Discussion
Compare the three models by finding the 5 most similar terms to each of the following terms: 'hamlet', 'cauldron', 'nature', 'spirit', 'general', and 'prythee'. Comment on how well each model captured the meaning of the word, and if there are multiple meanings, which meaning was given.


In [151]:
import pandas as pd

# Flatten results into list of dicts for DataFrame
table_data = []
for term, model_dict in results.items():
    for model_name, sims in model_dict.items():
        if isinstance(sims, list):  # Valid results
            for rank, (word, score) in enumerate(sims[:5], 1):
                table_data.append({
                    'Term': term,
                    'Model': model_name,
                    'Rank': rank,
                    'Similar Word': word,
                    'Similarity': f"{score:.3f}"
                })
        else:  # Missing vocab
            table_data.append({
                'Term': term,
                'Model': model_name,
                'Rank': 'N/A',
                'Similar Word': sims,
                'Similarity': 'N/A'
            })

df = pd.DataFrame(table_data)


In [152]:
df.head(15)


,Term,Model,Rank,Similar Word,Similarity
0,hamlet,Skip-gram (Shakespeare),1,polo,0.889
1,hamlet,Skip-gram (Shakespeare),2,lord,0.879
2,hamlet,Skip-gram (Shakespeare),3,alert,0.873
3,hamlet,Skip-gram (Shakespeare),4,ophelia,0.866
4,hamlet,Skip-gram (Shakespeare),5,were,0.848
5,hamlet,CBOW (Shakespeare),1,good,0.993
6,hamlet,CBOW (Shakespeare),2,polo,0.992
7,hamlet,CBOW (Shakespeare),3,sweet,0.991
8,hamlet,CBOW (Shakespeare),4,faith,0.989
9,hamlet,CBOW (Shakespeare),5,actor,0.989


Compare the three models by finding the cosine similarity between the following pairs of terms: ('brutus', 'murder'), ('lady macbeth', 'queen gertrude'), ('fortinbras', 'norway'), ('rome', 'norway'), ('ghost', 'spirit'), ('macbeth', 'hamlet'). Comment on how well each model captured the similarity between these terms, especially considering the data that each was trained on.


In [155]:
cosine_tests = [
    ['brutus', 'murder'],
    ['lady macbeth', 'queen gertrude'],
    ['fortinbras', 'norway'],
    ['rome', 'norway'],
    ['ghost', 'hamlet'],
]

models = [model, model_sg, glove_model]

for i in cosine_tests:
    w1, w2 = i
    print(f'\nCosine similarity test for: {w1} & {w2}')
    for m in models:
        if m == glove_model:
            try:
                print('\n glove model similarity:', m.similarity(w1, w2))
            except KeyError:
                print('KeyError')

        elif m == model:
            try:
                print('\n CBOW:', m.wv.similarity(w1, w2))
            except KeyError:
                print('KeyError')
            
        elif m == model_sg:
            try:
                print('\n Skip-gram:', m.wv.similarity(w1, w2))
            except KeyError:
                print('KeyError')

            


Cosine similarity test for: brutus & murder
KeyError
KeyError

 glove model similarity: 0.07364358

Cosine similarity test for: lady macbeth & queen gertrude
KeyError
KeyError
KeyError

Cosine similarity test for: fortinbras & norway

 CBOW: 0.9989291

 Skip-gram: 0.92274

 glove model similarity: -0.028961957

Cosine similarity test for: rome & norway

 CBOW: 0.99611264

 Skip-gram: 0.4841918

 glove model similarity: 0.28583667

Cosine similarity test for: ghost & hamlet

 CBOW: 0.9833922

 Skip-gram: 0.75300914

 glove model similarity: 0.52421004


Compare the three models by finding the 5 most similar terms to each of the following word vectors obtained via linear combination: 'denmark' + 'queen', 'scotland' + 'army' + 'general', 'father' - 'man' + 'woman', 'mother' - 'woman' + 'man'. Comment on how well each model described the ideas behind these word vectors.


In [170]:

lc = {
    'lc1': ['denmark', 'queen'],
    'lc2': ['scotland', 'army', 'general'],
    'lc3': ['father', 'man', 'woman'],
    'lc4': ['mother', 'woman', 'man']
}

results = []

for i in range(1, 5):
    value = lc.get(f'lc{i}')
    print(f"lc{i}: {value}")

    for c in value:
        try:
            v1 = model.wv[c]  # Get vector
            results.append((f'lc{i}', c, v1))  # Store with context
        except KeyError:
            print(f"'{c}' not in CBOW vocab")
            results.append((f'lc{i}', c, None))

results_sg = []

for i in range(1, 5):
    value = lc.get(f'lc{i}')
    print(f"lc{i}: {value}")

    for c in value:
        try:
            v1 = model_sg.wv[c]  # Get vector
            results_sg.append((f'lc{i}', c, v1))  # Store with context
        except KeyError:
            print(f"'{c}' not in CBOW vocab")
            results_sg.append((f'lc{i}', c, None))

results_glove = []

for i in range(1, 5):
    value = lc.get(f'lc{i}')
    print(f"lc{i}: {value}")

    for c in value:
        try:
            v1 = glove_model[c]  # Get vector
            results_glove.append((f'lc{i}', c, v1))  # Store with context
        except KeyError:
            print(f"'{c}' not in CBOW vocab")
            results_glove.append((f'lc{i}', c, None))



print(f"\nCollected {len(results)} vectors")
print(f"\nCollected {len(results_sg)} vectors")
print(f"\nCollected {len(results_glove)} vectors")

lc1: ['denmark', 'queen']
lc2: ['scotland', 'army', 'general']
lc3: ['father', 'man', 'woman']
lc4: ['mother', 'woman', 'man']
lc1: ['denmark', 'queen']
lc2: ['scotland', 'army', 'general']
lc3: ['father', 'man', 'woman']
lc4: ['mother', 'woman', 'man']
lc1: ['denmark', 'queen']
lc2: ['scotland', 'army', 'general']
lc3: ['father', 'man', 'woman']
lc4: ['mother', 'woman', 'man']

Collected 11 vectors

Collected 11 vectors

Collected 11 vectors


In [ ]:
def get_vectors(lc_name, results_list):
    """Extract vectors for lc1-4 from results list"""
    vec_dict = {}
    for lc, word, vec in results_list:
        if lc == lc_name and vec is not None:
            vec_dict[word] = vec
    return vec_dict

analogies = {
    'denmark + queen': ('lc1', lambda d: d['denmark'] + d['queen']),
    'scotland + army + general': ('lc2', lambda d: d['scotland'] + d['army'] + d['general']),
    'father - man + woman': ('lc3', lambda d: d['father'] - d['man'] + d['woman']),
    'mother - woman + man': ('lc4', lambda d: d['mother'] - d['woman'] + d['man'])
}

model_data = [
    ("CBOW (Shakespeare)", results, model),
    ("Skip-gram (Shakespeare)", results_sg, model_sg),
    ("GloVe (6B)", results_glove, glove_model)
]

comparison_table = []
for model_name, results_list, model_obj in model_data:
    print(f"\n=== {model_name} ===")
    
    for desc, (lc_name, compute_func) in analogies.items():
        vecs = get_vectors(lc_name, results_list)
        
        if all(word in vecs for word in lc[lc_name]):
            try:
                combo_vec = compute_func(vecs)
                
                # Find most similar using correct model object
                top5 = model_obj.most_similar(positive=[combo_vec], topn=5)
                
                print(f"{desc}:")
                for word, score in top5:
                    print(f"  {word}: {score:.3f}")
                    comparison_table.append({
                        'Analogy': desc, 
                        'Model': model_name, 
                        'Top Word': word, 
                        'Score': f"{score:.3f}"
                    })
            except Exception as e:
                print(f"{desc}: Error - {e}")
        else:
            print(f"{desc}: Missing words in {lc_name}")

# Final table
import pandas as pd
df = pd.DataFrame(comparison_table)
print("\n=== SUMMARY TABLE ===")
print(df.groupby(['Analogy', 'Model'])['Top Word'].first().unstack(fill_value='N/A'))
df.to_csv('linear_combination_results.csv', index=False)



=== CBOW (Shakespeare) ===
denmark + queen:
  ophelia: 0.998
  alert: 0.998
  guildensterne: 0.998
  macduffe: 0.998
  attendant: 0.998
scotland + army + general:
  general: 0.999
  yong: 0.999
  ear: 0.999
  valiant: 0.999
  liberty: 0.999
father - man + woman:
  father: 0.994
  oh: 0.984
  lost: 0.984
  mother: 0.983
  much: 0.982
mother - woman + man:
  mother: 0.994
  man: 0.992
  silence: 0.991
  bath: 0.991
  mouth: 0.991

=== Skip-gram (Shakespeare) ===
denmark + queen:
  queen: 0.966
  alert: 0.929
  denmark: 0.917
  qu: 0.915
  ophelia: 0.912
scotland + army + general:
  plebeian: 0.974
  space: 0.972
  bowl: 0.968
  flourish: 0.967
  titan: 0.967
father - man + woman:
  father: 0.843
  lost: 0.776
  deer: 0.715
  slain: 0.710
  mother: 0.702
mother - woman + man:
  mother: 0.791
  father: 0.663
  wife: 0.645
  man: 0.643
  brother: 0.642

=== GloVe (6B) ===
denmark + queen:
  queen: 0.859
  denmark: 0.836
  sweden: 0.736
  norway: 0.692
  princess: 0.689
scotland + army + ge

Give overall comments on how each model performs. Describe what data you would use to train a better word embedding model to captures the meaning of Shakespearean English.